In [ ]:
# === Common notebook preamble (load llm_math package) ===
import sys
from pathlib import Path

# Candidate paths for locating the llm_math package
_candidates = [
    '.', 'src', '..', '../src',
    '/content/llm-math-book/src',
    '/content/llm-math-book',
    '/workspace/src',
    '/workspace',
]
# Add parent directories as candidates when running from the notebooks folder
for p in Path.cwd().parents:
    _candidates.append(str(p / 'src'))
    _candidates.append(str(p))

for p in _candidates:
    if p and p not in sys.path and Path(p).exists():
        sys.path.insert(0, p)

# Try importing llm_math
try:
    from llm_math import viz, bench, data
    _LLM_MATH_OK = True
except ImportError as e:
    _LLM_MATH_OK = False
    print(f"[Warning] failed to load the llm_math package: {e}")
    print("  Clone the GitHub repository and run colab_setup.sh.")
# === End preamble ===


# Ch 10. Regularization Techniques — Generalization and Training Stability

> **Learning Objectives**
> - Understand mathematically why Dropout creates an ensemble effect
> - Explain the differences among BatchNorm, LayerNorm, and RMSNorm
> - Understand why LayerNorm/RMSNorm are used in LLMs

## 10.1 Overfitting and Generalization

**Overfitting**: the model fits the training data well but performs poorly on new data.
- Training loss ↓, validation loss ↑ pattern

Regularization constrains model complexity and improves generalization performance.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from llm_math.data import make_spiral_data, load_mnist_small

# Overfitting example: a large model on a small dataset
X_sp, y_sp = make_spiral_data(50, 3)
X_t = torch.tensor(X_sp, dtype=torch.float32)
y_t = torch.tensor(y_sp, dtype=torch.long)

torch.manual_seed(0)
big_model = nn.Sequential(
    nn.Linear(2, 256), nn.ReLU(),
    nn.Linear(256, 256), nn.ReLU(),
    nn.Linear(256, 256), nn.ReLU(),
    nn.Linear(256, 3)
)
opt = torch.optim.Adam(big_model.parameters(), lr=0.01)
loss_fn = nn.CrossEntropyLoss()

train_losses = []
for epoch in range(500):
    opt.zero_grad()
    out = big_model(X_t)
    loss = loss_fn(out, y_t)
    loss.backward()
    opt.step()
    train_losses.append(loss.item())

plt.figure(figsize=(8, 4))
plt.plot(train_losses, label='train loss')
plt.xlabel('Epoch'); plt.ylabel('Loss')
plt.title('Overfitting Model: training loss approaches 0, but generalization fails')
plt.legend(); plt.grid(True, alpha=0.3)
plt.yscale('log')
plt.tight_layout()
plt.savefig('../figures/ch10_overfitting.png', dpi=100, bbox_inches='tight')
plt.show()
print(f"Final training loss: {train_losses[-1]:.4f} (near 0)")
print("But performance will drop on new data.")


## 10.2 L1 and L2 Regularization — Bayesian Perspective

$$\mathcal{L}_{\text{reg}} = \mathcal{L} + \lambda \|\theta\|_2^2 \quad (L^2)$$
$$\mathcal{L}_{\text{reg}} = \mathcal{L} + \lambda \|\theta\|_1 \quad (L^1)$$

- $L^2$: keeps weights close to 0. From a Bayesian perspective, this is equivalent to a Gaussian prior $\theta \sim \mathcal{N}(0, \sigma^2)$.
- $L^1$: induces sparsity (some weights become exactly 0). Equivalent to a Laplace prior.


In [ ]:
# Effect of L2 regularization
torch.manual_seed(0)
model_no_reg = nn.Sequential(
    nn.Linear(784, 256), nn.ReLU(),
    nn.Linear(256, 10)
)
torch.manual_seed(0)
model_l2 = nn.Sequential(
    nn.Linear(784, 256), nn.ReLU(),
    nn.Linear(256, 10)
)

X_tr, y_tr, X_te, y_te = load_mnist_small(n_train=2000, n_test=1000)
X_tr_t = torch.tensor(X_tr, dtype=torch.float32)
y_tr_t = torch.tensor(y_tr, dtype=torch.long)
X_te_t = torch.tensor(X_te, dtype=torch.float32)
y_te_t = torch.tensor(y_te, dtype=torch.long)

loss_fn = nn.CrossEntropyLoss()
opt_no = torch.optim.SGD(model_no_reg.parameters(), lr=0.5)
opt_l2 = torch.optim.SGD(model_l2.parameters(), lr=0.5, weight_decay=0.01)

def train(model, opt, n_epochs=20):
    tr_losses, te_losses = [], []
    for _ in range(n_epochs):
        opt.zero_grad()
        out = model(X_tr_t)
        loss = loss_fn(out, y_tr_t)
        loss.backward()
        opt.step()
        tr_losses.append(loss.item())
        with torch.no_grad():
            te_loss = loss_fn(model(X_te_t), y_te_t).item()
            te_losses.append(te_loss)
    return tr_losses, te_losses

tr_no, te_no = train(model_no_reg, opt_no, n_epochs=15)
tr_l2, te_l2 = train(model_l2, opt_l2, n_epochs=15)

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(tr_no, 'b-', label='Train (no reg)', linewidth=2)
ax.plot(te_no, 'b--', label='Test (no reg)', linewidth=2)
ax.plot(tr_l2, 'r-', label='Train (L2)', linewidth=2)
ax.plot(te_l2, 'r--', label='Test (L2)', linewidth=2)
ax.set_xlabel('Epoch'); ax.set_ylabel('Loss')
ax.set_title('L2 Normalization Effect: Overfitting Decrease')
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('../figures/ch10_l2_regularization.png', dpi=100, bbox_inches='tight')
plt.show()


## 10.3 Dropout — The Ensemble Effect

During training, set neurons to 0 with probability $p$:
$$\tilde{\mathbf{h}} = \frac{\mathbf{m} \odot \mathbf{h}}{p}, \quad \mathbf{m}_i \sim \mathrm{Bernoulli}(1-p)$$

- Scale by $1/p$ to correct the expectation
- Do not use dropout at inference time (use all neurons)

Effect: train a different subnetwork each time → ensemble effect. Prevents co-adaptation between neurons.


In [ ]:
# Effect of dropout
torch.manual_seed(0)

def make_model(dropout=0.0):
    return nn.Sequential(
        nn.Linear(784, 512), nn.ReLU(), nn.Dropout(dropout),
        nn.Linear(512, 256), nn.ReLU(), nn.Dropout(dropout),
        nn.Linear(256, 10)
    )

results = {}
for dropout in [0.0, 0.3, 0.5]:
    torch.manual_seed(0)
    model = make_model(dropout)
    opt = torch.optim.Adam(model.parameters(), lr=0.001)
    tr_losses, te_accs = [], []
    for epoch in range(15):
        opt.zero_grad()
        out = model(X_tr_t)
        loss = loss_fn(out, y_tr_t)
        loss.backward()
        opt.step()
        tr_losses.append(loss.item())
        model.eval()
        with torch.no_grad():
            pred = model(X_te_t).argmax(1)
            acc = (pred == y_te_t).float().mean().item()
            te_accs.append(acc)
        model.train()
    results[dropout] = (tr_losses, te_accs)
    print(f"Dropout={dropout}: final test acc = {te_accs[-1]*100:.2f}%")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for d, (tr, te) in results.items():
    axes[0].plot(tr, label=f'dropout={d}', linewidth=2)
    axes[1].plot([a*100 for a in te], label=f'dropout={d}', linewidth=2)
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Train Loss')
axes[0].set_title('Effect of Dropout on Training Loss')
axes[0].legend(); axes[0].grid(True, alpha=0.3)
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Test Accuracy (%)')
axes[1].set_title('Effect of Dropout on Test Accuracy')
axes[1].legend(); axes[1].grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('../figures/ch10_dropout.png', dpi=100, bbox_inches='tight')
plt.show()


## 10.4 Batch Normalization (BatchNorm)

$$\hat{x} = \frac{x - \mu_B}{\sqrt{\sigma_B^2 + \epsilon}}, \quad y = \gamma \hat{x} + \beta$$

- $\mu_B, \sigma_B^2$: mini-batch statistics
- $\gamma, \beta$: learnable scale/shift
- Mitigates "internal covariate shift" (though newer explanations also exist)
- At inference: use learned moving averages

**Problem**: not suitable for RNNs/Transformers with variable sequence lengths. Unstable with small batches.


In [ ]:
# Visualize BatchNorm
np.random.seed(0)
# Activation-value distribution
import torch.nn.functional as F

torch.manual_seed(0)
x = torch.randn(64, 100) * 5 + 3  # mean 3, standard deviation 5
bn = nn.BatchNorm1d(100)
y = bn(x)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(x.flatten().numpy(), bins=50, alpha=0.7, color='blue', label='Before normalization')
axes[0].set_title(f'Before normalization: mean={x.mean():.2f}, std={x.std():.2f}')
axes[0].legend()
axes[1].hist(y.detach().flatten().numpy(), bins=50, alpha=0.7, color='red', label='After normalization')
axes[1].set_title(f'After BatchNorm: mean={y.mean():.2f}, std={y.std():.2f}')
axes[1].legend()
plt.tight_layout()
plt.savefig('../figures/ch10_batchnorm.png', dpi=100, bbox_inches='tight')
plt.show()


## 10.5 Layer Normalization (LayerNorm) — The Transformer Standard

$$\mu = \frac{1}{d}\sum_i x_i, \quad \sigma^2 = \frac{1}{d}\sum_i (x_i - \mu)^2$$
$$\hat{x} = \frac{x - \mu}{\sqrt{\sigma^2 + \epsilon}}, \quad y = \gamma \hat{x} + \beta$$

Difference:
- **BatchNorm**: normalize over the batch dimension (per feature)
- **LayerNorm**: normalize over the feature dimension (per sample)

LayerNorm works independently of batch size → suitable for sequence models. **LLM standard**.


In [ ]:
# Compare BatchNorm vs LayerNorm
torch.manual_seed(0)
# (batch, features) = (4, 6)
x = torch.tensor([
    [1., 2., 3., 4., 5., 6.],
    [10., 20., 30., 40., 50., 60.],
    [-1., -2., -3., -4., -5., -6.],
    [0.1, 0.2, 0.3, 0.4, 0.5, 0.6],
])

print("Input (batch=4, features=6):")
print(x)

# BatchNorm: statistics by feature (column-wise)
bn = nn.BatchNorm1d(6, affine=False)
y_bn = bn(x)
print(f"\nBatchNorm (normalization by feature/column):")
print(y_bn.round(decimals=3))
print(f"  each column has mean ≈ 0, std ≈ 1")

# LayerNorm: statistics by sample (row-wise)
ln = nn.LayerNorm(6, elementwise_affine=False)
y_ln = ln(x)
print(f"\nLayerNorm (normalization by sample/row):")
print(y_ln.round(decimals=3))
print(f"  each row has mean ≈ 0, std ≈ 1")


## 10.6 RMSNorm — A Simplification of LayerNorm

$$\bar{x} = \frac{x}{\sqrt{\frac{1}{d}\sum_i x_i^2 + \epsilon}} \odot \gamma$$

Skip mean subtraction and normalize only by RMS (root mean square). **The choice used by LLaMA**.

Advantages:
- Slightly less computation
- Performance is similar to LayerNorm
- Simpler implementation


In [ ]:
# Implement RMSNorm
class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-6):
        super().__init__()
        self.gamma = nn.Parameter(torch.ones(dim))
        self.eps = eps

    def forward(self, x):
        rms = torch.sqrt(torch.mean(x**2, dim=-1, keepdim=True) + self.eps)
        return x / rms * self.gamma

# Comparison
torch.manual_seed(0)
x = torch.randn(2, 4, 8) * 3 + 1  # (batch, seq, dim)

ln = nn.LayerNorm(8)
rmsn = RMSNorm(8)

y_ln = ln(x)
y_rmsn = rmsn(x)

print(f"Input: shape={x.shape}, mean={x.mean():.4f}, std={x.std():.4f}")
print(f"LayerNorm: mean={y_ln.mean():.4f}, std={y_ln.std():.4f}")
print(f"RMSNorm:   mean={y_rmsn.mean():.4f}, std={y_rmsn.std():.4f}")
print("\n=> LayerNorm sets mean=0, while RMSNorm does not force the mean to 0 (simplification)")


## 10.7 Gradient Norm Clipping

$$\mathbf{g} \leftarrow \min\left(1, \frac{c}{\|\mathbf{g}\|}\right) \mathbf{g}$$

Essential in LLM training. Prevents loss spikes and divergence.


In [ ]:
# PyTorch gradient clipping
torch.manual_seed(0)
model = nn.Linear(10, 5)
x = torch.randn(3, 10)
y = torch.randn(3, 5)

loss = nn.MSELoss()(model(x), y)
loss.backward()

# before clipping
grad_norm_before = sum((p.grad**2).sum() for p in model.parameters()).sqrt().item()
torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
grad_norm_after = sum((p.grad**2).sum() for p in model.parameters()).sqrt().item()

print(f"Gradient Norm:")
print(f"  before clipping: {grad_norm_before:.4f}")
print(f"  after clipping: {grad_norm_after:.4f} (limited to 1.0)")


## 10.8 Key Takeaways

| Technique | Formula | Use |
|---|---|---|
| $L^2$ regularization | $\mathcal{L} + \lambda\|\theta\|^2$ | Keep weights small |
| $L^1$ regularization | $\mathcal{L} + \lambda\|\theta\|_1$ | Sparsity |
| Dropout | $\frac{\mathbf{m} \odot \mathbf{h}}{p}$ | Ensemble effect |
| BatchNorm | $\frac{x - \mu_B}{\sigma_B}$ | Batch-wise normalization |
| LayerNorm | $\frac{x - \mu}{\sigma}$ (per feature) | LLM standard |
| RMSNorm | $\frac{x}{\sqrt{\frac{1}{d}\sum x_i^2}} \gamma$ | LLaMA standard |

## Exercises

1. Compare MNIST MLP test accuracy for Dropout $p=0, 0.2, 0.5, 0.8$.
2. Apply BatchNorm and LayerNorm to the same MLP and compare learning curves.
3. Compare RMSNorm and LayerNorm in a small Transformer (implemented in the next chapter).
4. Compare learning curves for weight decay $\lambda = 0, 10^{-4}, 10^{-2}, 10^{-1}$.
5. Experiment with how gradient clipping affects training stability at a large learning rate.

> Solutions: `solutions/ch10_solutions.ipynb`
